# CO2 Emissions Forecasting (R3)

This notebook follows `Fix.md` **R3 only** (no EDA/PCA/clustering sections).

## Research Question
How accurately can future `co2_per_capita` be forecast from historical economic, demographic, and energy-related variables, and do advanced models outperform baselines?

## Workflow
1. Build lag and trend features
2. Use strict time-aware split
3. Train baseline, tree, neural, and sequence models
4. Attempt one Hugging Face pretrained model
5. Build ensemble and export comparison table


In [ ]:
# Purpose: Import all packages needed for R3 modeling, evaluation, and plotting.
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

# Optional package: XGBoost baseline runs only if import works.
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# Optional backend for LSTM.
import torch
import torch.nn as nn

print('xgboost available:', HAS_XGB)


In [ ]:
# Purpose: Define paths, target, base features, and load cleaned panel data.
DATA_PATH = Path('cleaned_co2_data_20vars.csv')
OUTPUT_DIR = Path('output')
(OUTPUT_DIR / 'tables').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)

TARGET = 'co2_per_capita'
BASE_FEATURES = [
    'gdp', 'population', 'primary_energy_consumption', 'energy_per_capita', 'energy_per_gdp',
    'co2', 'total_ghg', 'consumption_co2', 'methane', 'nitrous_oxide',
    'cement_co2', 'flaring_co2', 'trade_co2', 'coal_co2', 'gas_co2', 'oil_co2'
]

df = pd.read_csv(DATA_PATH).sort_values(['country', 'year']).reset_index(drop=True)
print('Shape:', df.shape)
print('Years:', int(df['year'].min()), '-', int(df['year'].max()))


In [ ]:
# Purpose: Define helper functions for metrics and country-wise lag/trend features.
def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if len(y_true) == 0 or len(y_pred) == 0:
        return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }

def metrics_str(m):
    if np.isnan(m['RMSE']):
        return 'RMSE=N/A MAE=N/A R2=N/A'
    return f"RMSE={m['RMSE']:.4f} MAE={m['MAE']:.4f} R2={m['R2']:.4f}"

def add_time_features(panel_df):
    # Country-wise grouping preserves temporal order and avoids future leakage.
    g = panel_df.groupby('country', group_keys=False)
    out = panel_df.copy()

    # Required lag features from Fix.md.
    out['co2_lag1'] = g[TARGET].shift(1)
    out['co2_lag3'] = g[TARGET].shift(3)
    out['gdp_lag1'] = g['gdp'].shift(1)
    out['energy_lag1'] = g['energy_per_capita'].shift(1)

    # Trend feature from prior values only.
    out['co2_roll3_mean'] = g[TARGET].transform(lambda s: s.shift(1).rolling(3).mean())
    return out


In [ ]:
# Purpose: Engineer lag features, apply strict time split, and scale predictors with train-only stats.
lag_df = add_time_features(df)
FORECAST_FEATURES = BASE_FEATURES + ['co2_lag1', 'co2_lag3', 'gdp_lag1', 'energy_lag1', 'co2_roll3_mean']
lag_df = lag_df.dropna(subset=FORECAST_FEATURES + [TARGET]).copy()

train = lag_df[(lag_df['year'] >= 1990) & (lag_df['year'] <= 2015)].copy()
val   = lag_df[(lag_df['year'] >= 2016) & (lag_df['year'] <= 2019)].copy()
test  = lag_df[(lag_df['year'] >= 2020) & (lag_df['year'] <= 2022)].copy()

print('Lag split sizes:', len(train), len(val), len(test))
print('Feature count:', len(FORECAST_FEATURES))

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(train[FORECAST_FEATURES]), columns=FORECAST_FEATURES, index=train.index)
X_val   = pd.DataFrame(scaler.transform(val[FORECAST_FEATURES]), columns=FORECAST_FEATURES, index=val.index)
X_test  = pd.DataFrame(scaler.transform(test[FORECAST_FEATURES]), columns=FORECAST_FEATURES, index=test.index)

y_train = train[TARGET].values
y_val = val[TARGET].values
y_test = test[TARGET].values


In [ ]:
# Purpose: Train baseline, tree, and MLP models and collect test predictions.
pred_test = {}

for name, model in {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=1.0, random_state=42),
    'Lasso': Lasso(alpha=0.001, random_state=42, max_iter=5000),
}.items():
    model.fit(X_train, y_train)
    pred_test[name] = model.predict(X_test)
    print(name, 'test ->', metrics_str(evaluate(y_test, pred_test[name])))

rf = RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_test['RandomForest'] = rf.predict(X_test)
print('RandomForest test ->', metrics_str(evaluate(y_test, pred_test['RandomForest'])))

if HAS_XGB:
    xgb = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        objective='reg:squarederror', random_state=42, n_jobs=-1
    )
    xgb.fit(X_train, y_train)
    pred_test['XGBoost'] = xgb.predict(X_test)
    print('XGBoost test ->', metrics_str(evaluate(y_test, pred_test['XGBoost'])))
else:
    print('XGBoost skipped: package not installed.')

mlp = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', max_iter=400, random_state=42)
mlp.fit(X_train, y_train)
pred_test['MLP'] = mlp.predict(X_test)
print('MLP test ->', metrics_str(evaluate(y_test, pred_test['MLP'])))


In [ ]:
# Purpose: Train a compact LSTM on country-wise sequences.
SEQ_LEN = 5

def build_sequences(df_split, feature_cols, target_col, seq_len=5):
    X_list, y_list = [], []
    for _, g in df_split.sort_values(['country', 'year']).groupby('country'):
        Xg = g[feature_cols].values
        yg = g[target_col].values
        if len(g) <= seq_len:
            continue
        for i in range(seq_len, len(g)):
            X_list.append(Xg[i-seq_len:i])
            y_list.append(yg[i])
    if len(X_list) == 0:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,))
    return np.asarray(X_list, dtype=np.float32), np.asarray(y_list, dtype=np.float32)

X_seq_train, y_seq_train = build_sequences(train, FORECAST_FEATURES, TARGET, SEQ_LEN)
X_seq_test, y_seq_test = build_sequences(test, FORECAST_FEATURES, TARGET, SEQ_LEN)
print('LSTM sequence shapes:', X_seq_train.shape, X_seq_test.shape)

if len(y_seq_train) > 0 and len(y_seq_test) > 0:
    class LSTMReg(nn.Module):
        def __init__(self, n_in, hidden=64):
            super().__init__()
            self.lstm = nn.LSTM(input_size=n_in, hidden_size=hidden, batch_first=True)
            self.fc = nn.Linear(hidden, 1)
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :]).squeeze(-1)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTMReg(X_seq_train.shape[2], hidden=64).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    Xtr = torch.tensor(X_seq_train, device=device)
    ytr = torch.tensor(y_seq_train, device=device)
    Xte = torch.tensor(X_seq_test, device=device)

    for _ in range(120):
        opt.zero_grad()
        loss = loss_fn(model(Xtr), ytr)
        loss.backward()
        opt.step()

    with torch.no_grad():
        pred_test['LSTM_seq'] = model(Xte).cpu().numpy()
    print('LSTM test ->', metrics_str(evaluate(y_seq_test, pred_test['LSTM_seq'])))
else:
    print('LSTM skipped: not enough sequence rows.')


In [ ]:
# Purpose: Hugging Face pretrained-model attempt, ensemble, final comparison export, and plot.
hf_status = 'not_run'
try:
    from transformers import PatchTSTForPrediction
    _ = PatchTSTForPrediction.from_pretrained('ibm/patchtst-etth1-pretrain')
    hf_status = 'loaded_patchtst_checkpoint'
except Exception as e:
    hf_status = f'skipped ({str(e)[:120]})'
print('Hugging Face status:', hf_status)

rows = []
for name, p in pred_test.items():
    if name == 'LSTM_seq':
        rows.append({'model': name, **evaluate(y_seq_test, p)})
    else:
        rows.append({'model': name, **evaluate(y_test, p)})

pool = [m for m in ['Linear', 'Ridge', 'Lasso', 'RandomForest', 'XGBoost', 'MLP'] if m in pred_test]
if len(pool) >= 2:
    ens = np.mean(np.column_stack([pred_test[m] for m in pool]), axis=1)
    rows.append({'model': 'Ensemble_avg', **evaluate(y_test, ens)})

comp_df = pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)
comp_df.to_csv(OUTPUT_DIR / 'tables' / 'model_comparison.csv', index=False)
print('Best RMSE model:', comp_df.loc[0, 'model'])
display(comp_df)

plt.figure(figsize=(10, 4))
plot_df = comp_df.dropna(subset=['RMSE'])
plt.bar(plot_df['model'], plot_df['RMSE'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('RMSE')
plt.title('R3 Forecasting Model Comparison')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'r3_model_comparison_rmse.png', dpi=150)
plt.show()
